# CAS fork of document templates in FAIR Wizard / DSW
New code for CAS fork with ASEP integrations, forked from **Horizon Europe DMP 3.1.2** (`fw.common-horizon-europe-dmp`, requires `dsw.root` / `dsw.lifesciences` 2.8.1+).

Lines indicate the lines in the original code (3.1.2) that should be replaced by the new code.

## Document Templates:
- Horizon Europe DMP
- MŠMT DMP
- TAČR DMP
- GAČR DMP
- AZV ČR DMP


## reference.docx
`src/word/reference.docx` - footer: "Data Management Plan created in FAIR Wizard." + change logo to FAIR Wizard
(not required if forked from a version from FAIR Wizard registry)

## src/macros.html.j2
Added new macros for CAS integration.

**insert after line 1** (`{# Formatting macros #}`):

In [ ]:
{# CAS - integration reply (ASEP); also works for a reply typed as plain text.
   lines=false: inline HTML (paragraphs joined) - for names inside <strong>, <h3> and sentences.
   lines=true: lines of the reply separated by <br> (a selected item keeps its markdown, a typed text is bold). #}
{%- macro integrationCAS(o, lines=false) -%}
  {%- if o._value is not none -%}
    {%- if lines and '_raw' not in o -%}
      <strong>{{ o._value }}</strong>
    {%- elif lines -%}
      {{ o._value|markdown|replace("</p>\n<p>", "<br>"|safe)|replace("</p><p>", "<br>"|safe)|replace("<p>", "")|replace("</p>", "")|trim }}
    {%- else -%}
      {{ o._value|markdown|replace("<p>", "")|replace("</p>", "")|trim }}
    {%- endif -%}
  {%- endif -%}
{%- endmacro -%}

{# CAS - only the name (first line) of an integration reply as inline HTML - for sentences.
   A reply typed as plain text is returned in bold, same as a selected item. #}
{%- macro integrationCASName(o) -%}
  {%- if o._value is not none -%}
    {%- if '_raw' in o -%}
      {{ o._value.strip().replace("\r\n", "\n").split("\n")[0]|markdown|replace("<p>", "")|replace("</p>", "")|trim }}
    {%- else -%}
      <strong>{{ o._value }}</strong>
    {%- endif -%}
  {%- endif -%}
{%- endmacro -%}

{# CAS - only the name of a project from an integration reply as inline HTML - for sentences.
   Like integrationCASName, but a trailing " - <project number>" on the first line is cut off
   (the number is a link, or a single word with a digit). A reply typed as plain text is returned in bold. #}
{%- macro integrationCASProjectName(o) -%}
  {%- if o._value is not none -%}
    {%- if '_raw' in o -%}
      {%- set first = o._value.strip().replace("\r\n", "\n").split("\n")[0] -%}
      {%- set parts = first.rsplit(" - ", 1) -%}
      {%- if parts|length == 2 -%}
        {%- set tail = parts[1]|trim -%}
        {%- if "](" in tail or (" " not in tail and tail|length <= 25 and tail|select("in", "0123456789")|list|length > 0) -%}
          {%- set first = parts[0] -%}
        {%- endif -%}
      {%- endif -%}
      {{ first|markdown|replace("<p>", "")|replace("</p>", "")|trim }}
    {%- else -%}
      <strong>{{ o._value }}</strong>
    {%- endif -%}
  {%- endif -%}
{%- endmacro -%}


## src/contributors.html.j2
(also used for MS Word - there is no separate `src/word/contributors.html.j2` any more)

**lines 3-5:**

In [ ]:
{%- for contributor in x.administrativeInformation.contributors if contributor.name._value -%}
  {#- CAS - the name is an integration question (macros.integrationCAS), lines=true keeps its lines -#}
  {%- do contributors.append({
    'name': macros.integrationCAS(contributor.name, true),

**lines 18-37** (original numbering - the `<ul>` ... `</ul>` of the list; the `{%- elif dc.project.created_by -%}` branch below stays unchanged)

CAS layout: one paragraph per contributor, lines separated by `<br>` (no paragraph gaps): name, affiliation, ORCID + e-mail, roles. A name selected from the integration already contains the lines (name, affiliation, ORCID, e-mail); a name typed by hand is bold and followed by the separate affiliation / ORCID / e-mail answers, so both look the same. Only the name is bold (in HTML/PDF and in MS Word) - see `.contributor .name` in style.css.

In [ ]:
  {# new code for CAS fork - layout of the list of contributors #}
  <ul>
  {% for contributor in contributors %}
    <li class="contributor">
      <span class="name">
        {{- contributor.name -}}
        {%- if contributor.affiliation -%}
          <br>{{ macros.ror(contributor.affiliation) }}
        {%- endif -%}
        {%- if contributor.orcid and contributor.email -%}
          <br>{{ macros.orcid(contributor.orcid) }} {{ macros.formatEmail(contributor.email) }}
        {%- elif contributor.orcid -%}
          <br>{{ macros.orcid(contributor.orcid) }}
        {%- elif contributor.email -%}
          <br>{{ macros.formatEmail(contributor.email) }}
        {%- endif -%}
      </span>
      {%- if contributor.roles|length > 0 -%}
        <br>{% if contributor.roles|length == 1 %}{% trans %}Role{% endtrans %}{% else %}{% trans %}Roles{% endtrans %}{% endif %}: {% for role in contributor.roles %}<span class="role">{{ role }}</span>{% if not loop.last %}, {% endif %}{% endfor %}
      {%- endif %}
    </li>
  {% endfor %}
  </ul>

## src/projects.html.j2
(also used for MS Word - there is no separate `src/word/projects.html.j2` any more)

**line 12:**

In [ ]:
    {%- set projectName = macros.integrationCAS(project.name) -%} {#- CAS - integration question -#}

## src/questions/01-data-summary.html.j2
Deleted distinctions between reference and non-reference datasets

**line 128**

In [ ]:
          <p>{% trans %}We identified the following datasets considered for potential re-use:{% endtrans %}</p> {# CAS - deleted "reference" #}

**line 227**

In [ ]:
          {# CAS - deleted sentence about non-reference datasets #}

## src/questions/04-resource-allocation.html.j2
**line 14** (project name in the list of costs; used on the following line 18, which stays unchanged)

In [ ]:
            {%- set projectItemNameReply = macros.integrationCASProjectName(project.name) -%} {#- CAS - integration question, only the name -#}

**line 18** (no project number after the name - it comes from the separate `project.id` answer; `projectItemNumberReply` on line 16 is then unused and can stay):

In [ ]:
                <strong>{{ projectItemNameReply if projectItemNameReply else _("(no name given)") }}</strong> {#- CAS - without the project number -#}

**line 251** (names of contributors in the responsibility sentences - only the name, without affiliation, ORCID and e-mail; lines 262-276 stay unchanged)

In [ ]:
        {%- set name = macros.integrationCASName(contributor.name) -%} {#- CAS - integration question, only the name -#}

## src/questions/06-ethics.html.j2
**line 14** (used in the condition on line 16 and in `'name': projectName` on line 45, both stay unchanged)

In [ ]:
      {%- set projectName = macros.integrationCASProjectName(projectItem.name) -%} {#- CAS - integration question, only the name -#}

## src/style.css
Bullets in the list of contributors (3.1.2 removes them; MS Word does not use this stylesheet, so it is not affected) and bold only for the name (HTML/PDF).

**lines 229-232:**

In [ ]:
#dmp-contributors ul {
  list-style: disc; /* CAS - bullets in the list of contributors */
  padding-left: 2em;
}

**lines 239-241** (only the name is bold, as in MS Word; the name itself is in `<strong>` / markdown bold):

In [ ]:
.contributor .name {
  font-weight: normal; /* CAS - only the name (<strong>) is bold, same as in MS Word */
}